In [ ]:
# SECTION 1 — Imports and configuration
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, precision_score, recall_score
from torch.optim.lr_scheduler import ReduceLROnPlateau

import warnings
warnings.filterwarnings('ignore')

# REPRODUCIBILITY
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# KAGGLE SAFETY: Print device info
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("CUDA availability:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# Paths
checkpoint_path = '/kaggle/working/vitb16_best.pth'
history_path = '/kaggle/working/vitb16_training_history.csv'


In [ ]:
# Dataset
base_path = "/kaggle/input/datasets/manan1717/ocular-disease-dataset/preprocessed"

total = 0
bad = []

if os.path.exists(base_path):
    print("dataset path:", base_path)
    for cls in sorted(os.listdir(base_path)):
        folder = os.path.join(base_path, cls)
        if not os.path.isdir(folder):
            continue
        for filename in os.listdir(folder):
            path = os.path.join(folder, filename)
            total += 1
            try:
                with Image.open(path) as img:
                    img.verify()
            except Exception as e:
                bad.append((filename, str(e)))

    print("number of images:", total)
    print("Bad/unreadable images:", len(bad))
else:
    print(f"Dataset path {base_path} not found. Please ensure it's attached.")


In [ ]:
# SECTION 3 — Class mapping
class_to_idx = {
    "N": 0,
    "D": 1,
    "G": 2,
    "C": 3,
    "A": 4,
    "H": 5,
    "M": 6
}
idx_to_class = {v: k for k, v in class_to_idx.items()}
classes = ["N", "D", "G", "C", "A", "H", "M"]

data = []
if os.path.exists(base_path):
    for label in classes:
        folder = os.path.join(base_path, label)
        if os.path.isdir(folder):
            for filename in os.listdir(folder):
                data.append({
                    "filename": filename,
                    "label": label,
                    "path": os.path.join(folder, filename)
                })

df = pd.DataFrame(data)
print("\nclass distribution:")
if len(df) > 0:
    print(df["label"].value_counts()[classes])


In [ ]:
# Split
if len(df) > 0:
    train_df, temp_df = train_test_split(
        df,
        test_size=0.20,
        stratify=df["label"],
        random_state=42
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        stratify=temp_df["label"],
        random_state=42
    )
    
    # Reset indices
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print("train/validation/test counts:")
    print("Train:", len(train_df))
    print("Validation:", len(val_df))
    print("Test:", len(test_df))
    
    # Verify no overlap
    train_paths = set(train_df['path'])
    val_paths = set(val_df['path'])
    test_paths = set(test_df['path'])
    
    print("Train/Val overlap:", len(train_paths.intersection(val_paths)))
    print("Train/Test overlap:", len(train_paths.intersection(test_paths)))
    print("Val/Test overlap:", len(val_paths.intersection(test_paths)))


In [ ]:
# Preprocessing
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
# SECTION 6 — Dataset and DataLoaders
class OcularDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        label = class_to_idx[row["label"]]

        if self.transform:
            image = self.transform(image)

        return image, label

if len(df) > 0:
    train_dataset = OcularDataset(train_df, train_transform)
    val_dataset = OcularDataset(val_df, eval_transform)
    test_dataset = OcularDataset(test_df, eval_transform)

    batch_size = 64

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    print("DataLoaders created.")


In [ ]:
# ViT-B16
if len(df) > 0:
    model = torchvision.models.vit_b_16(weights=torchvision.models.ViT_B_16_Weights.IMAGENET1K_V1)

    # Replace the classification head for 7 classes
    model.heads.head = nn.Linear(model.heads.head.in_features, 7)
    model = model.to(device)

    # Compute class weights for training
    train_labels = train_df["label"].values
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array(classes),
        y=train_labels
    )
    class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

    scaler = torch.amp.GradScaler('cuda')

    print("ViT-B/16 model, criterion, optimizer, and scheduler ready.")


In [ ]:
# Training
def train_epoch(model, dataloader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

def evaluate(model, dataloader, criterion, device):
    model.eval()
    val_loss_total = 0.0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating", leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss_total += loss.item() * images.size(0)
            predictions = outputs.argmax(dim=1)

            val_preds.extend(predictions.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss = val_loss_total / len(dataloader.dataset)
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds, average="macro")

    return val_loss, val_acc, val_f1, val_preds, val_labels


In [ ]:
# SECTION 9 — Training loop with checkpoint saving
if len(df) > 0:
    epochs = 10
    best_val_f1 = 0.0
    best_epoch = -1
    
    history = {
        "epoch": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "val_f1": [],
        "lr": []
    }

            
    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")
        
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler, device)
        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
        
        current_lr = optimizer.param_groups[0]['lr']
        
        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)
        history["lr"].append(current_lr)
        
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val Macro F1: {val_f1:.4f}")
        
        scheduler.step(val_f1)
        
        # Checkpoint saving
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch
            print(f"⭐ New best validation Macro F1: {best_val_f1:.4f}. Saving checkpoint...")
            
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_macro_f1': best_val_f1,
                'class_names': classes
            }
            torch.save(checkpoint, checkpoint_path)
            
        pd.DataFrame(history).to_csv(history_path, index=False)


In [ ]:
# SECTION 10 — Restore best checkpoint
if len(df) > 0:
    if os.path.exists(checkpoint_path):
        print(f"Loading best checkpoint...")
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Loaded checkpoint with Validation Macro F1: {checkpoint['best_val_macro_f1']:.4f}")
    else:
        print("Checkpoint not found!")

# Paths
checkpoint_path = '/kaggle/working/vitb16_best.pth'
history_path = '/kaggle/working/vitb16_training_history.csv'


In [ ]:
# Evaluation
if len(df) > 0:
    print("Evaluating on test set...")
    test_loss, test_acc, test_macro_f1, test_preds, test_labels = evaluate(model, test_loader, criterion, device)

    test_macro_prec = precision_score(test_labels, test_preds, average="macro")
    test_macro_rec = recall_score(test_labels, test_preds, average="macro")
    test_weighted_f1 = f1_score(test_labels, test_preds, average="weighted")
    
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Macro Precision: {test_macro_prec:.4f}")
    print(f"Test Macro Recall: {test_macro_rec:.4f}")
    print(f"Test Macro F1: {test_macro_f1:.4f}")
    print(f"Test Weighted F1: {test_weighted_f1:.4f}")


In [ ]:
# Results
if len(df) > 0:
    print("\nClassification Report:")
    print(classification_report(test_labels, test_preds, target_names=classes, digits=4))


In [ ]:
# SECTION 13 — Confusion matrix
if len(df) > 0:
    cm = confusion_matrix(test_labels, test_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title("ViT-B/16 Test Confusion Matrix")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

# Paths
checkpoint_path = '/kaggle/working/vitb16_best.pth'
history_path = '/kaggle/working/vitb16_training_history.csv'


In [ ]:
# SECTION 14 — Training curves
if len(df) > 0 and os.path.exists(history_path):
    hist_df = pd.read_csv(history_path)
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss', marker='o')
    plt.plot(hist_df['epoch'], hist_df['val_loss'], label='Val Loss', marker='o')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(hist_df['epoch'], hist_df['train_acc'], label='Train Acc', marker='o')
    plt.plot(hist_df['epoch'], hist_df['val_acc'], label='Val Acc', marker='o')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(6, 5))
    plt.plot(hist_df['epoch'], hist_df['val_f1'], label='Val Macro F1', marker='o', color='green')
    plt.title('Validation Macro F1 Score')
    plt.xlabel('Epoch')
    plt.ylabel('Macro F1')
    plt.legend()
    plt.grid(True)
    plt.show()

# Paths
checkpoint_path = '/kaggle/working/vitb16_best.pth'
history_path = '/kaggle/working/vitb16_training_history.csv'


In [ ]:
# SECTION 15 — Per-class F1
if len(df) > 0:
    class_f1 = f1_score(test_labels, test_preds, average=None)
    
    plt.figure(figsize=(8, 5))
    sns.barplot(x=classes, y=class_f1, palette='viridis')
    plt.title('Per-Class F1 Score on Test Set')
    plt.xlabel('Class')
    plt.ylabel('F1 Score')
    plt.ylim(0, 1)
    for i, v in enumerate(class_f1):
        plt.text(i, v + 0.02, f"{v:.4f}", ha='center', va='bottom', fontsize=9)
    plt.show()

# Paths
checkpoint_path = '/kaggle/working/vitb16_best.pth'
history_path = '/kaggle/working/vitb16_training_history.csv'


In [ ]:
# SECTION 16 — External image prediction
def predict_image(img_path, model, transform, device):
    if not os.path.exists(img_path):
        return None
    image = Image.open(img_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            outputs = model(image_tensor)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            pred_idx = probs.argmax(dim=1).item()
            pred_prob = probs[0, pred_idx].item()
    return classes[pred_idx], pred_prob

# Paths
checkpoint_path = '/kaggle/working/vitb16_best.pth'
history_path = '/kaggle/working/vitb16_training_history.csv'


In [ ]:
# SECTION 17 — Final checkpoint verification
if len(df) > 0:
    print("\nVIT-B16 EXPERIMENT COMPLETE")
    try:
    print(f"Best validation Macro F1: {best_val_f1:.4f}")
    print(f"Best epoch: {best_epoch}")
except NameError:
    print("Training variables not in memory (Training was skipped).")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Macro F1: {test_macro_f1:.4f}")
    print(f"Test Precision: {test_macro_prec:.4f}")
    print(f"Test Recall: {test_macro_rec:.4f}")
    if os.path.exists(checkpoint_path):
        print(f"Checkpoint path: {checkpoint_path}")
        print(f"Checkpoint file size: {os.path.getsize(checkpoint_path) / (1024*1024):.2f} MB")
    if os.path.exists(history_path):
        print(f"Training history path: {history_path}")

# Paths
checkpoint_path = '/kaggle/working/vitb16_best.pth'
history_path = '/kaggle/working/vitb16_training_history.csv'


In [ ]:
# External Validation
# TODO: Future validation on external hospital dataset.
# 1. Load completely separate dataset
# 2. Apply same eval_transform
# 3. Run predictions without retraining
# 4. Compare domain shift metrics
